In [ ]:
%load_ext autoreload
%autoreload 2


import earthshine_io as eio
import diagnostics_v2 as diag


In [ ]:
# 'data' is the subdirectory to look in. 
!python earthshine_io.py data --rebuild


In [ ]:
# Pull out subsets

cat = eio.read_catalog("data")

# Just a pandas command
hists = cat.query("stage == 'derived' and kind == 'depth_histogram'")

hists

In [ ]:
# Column names

for col in cat.columns:
    print(col)

In [ ]:
# Sometimes easier to see subsets
cat = eio.read_catalog("data")
cat[["dm_model","mDM_min","mDM_max","depth_min","depth_max","disk_radius","n_rows"]]


In [ ]:
# Documentation on load
? eio.load

In [ ]:
cat['dm_model'].unique()

In [ ]:
# 'combos' lets us chain selections together

cat = eio.read_catalog("data")

#eio.combos(cat, "mDM", dm_model="core")            # unique mDM_min/mDM_max combos
#eio.combos(cat, "depth", dm_model="core")          # unique depth combos
#eio.combos(cat)                                    # full run matrix, everything

# This is probably the most common
#eio.combos(cat, "depth", "mDM", "disk_radius", 'inner_detector_radius', 'inner_detector_half_len', dm_model="core")   # joint combos of both
eio.combos(cat, "depth", "mDM", "disk_radius", 'inner_detector_radius', 'inner_detector_half_len', dm_model="momentum_constrained")   # joint combos of both


In [ ]:
# Any of the plot commands will return a dataframe and the parameters

#model = 'floating'
model = 'momentum_constrained'
#model = 'core'

mass = 900000

mDM_min = mass
mDM_max = mass

#mDM_min = 2000
#mDM_max = 2000

depth_min=-4000.0
depth_max = -8.0
disk_radius = 4000.0

inner_detector_radius=1.0
inner_detector_half_len=2.50

emin = 100

#mass = 2000
#masses = [2000, 9000, 90000, 900000]

#inner_detector = False
inner_detector = True

###################################################################################################################
# Grab the data
###################################################################################################################

df, params = eio.load_many(cat, masses=[mass], dm_model=model, stage="combined",
                      depth_min=depth_min, depth_max=depth_max, disk_radius=disk_radius, \
                     inner_detector_radius=inner_detector_radius, inner_detector_half_len=inner_detector_half_len, \
                     mDM_min=mDM_min, mDM_max=mDM_max, eloss='ave')   # add filters until exactly 1 matches


#print(params)

# For now, save the dataframe
outfilename = f'kinematic_distributions_for_specific_masses_dm_model_{model}_vol_radius_{int(disk_radius)}_ID_r_{inner_detector_radius}_ID_hl_{inner_detector_half_len}_mx_{mass}.parquet'
print(f'Saving to {outfilename}...')
df.to_parquet(outfilename)
print(f'Saved')

dftmp,figs = diag.e_and_pt_plots(df, params, min_efinal=emin, inner_detector=inner_detector, plotdir="plots")

dftmp,figs = diag.origin_plots(df,  params, min_efinal=emin, inner_detector=inner_detector, plotdir="plots")


In [ ]:
df.columns

In [ ]:
df

In [ ]:
import json

pretty_json = json.dumps(params, indent=4)
print(pretty_json)

In [ ]:
for col in df.columns:
    print(col)

In [ ]:
df